# YOLO Object Detection - Discovery Challenge**Estimated Time**: 90 minutes**Course**: AI-2 Artificial Intelligence Foundations**Activity**: 17Harness the power of real-time object detection with YOLO (You Only Look Once)! This activity introduces YOLOv8 - a state-of-the-art computer vision model that detects and classifies 80 different object types in images and video streams.

## 📍 Where to run this — environment notes

> **🛠️ Stack:** Python + [Ultralytics YOLO](https://docs.ultralytics.com/) — `pip install ultralytics` (it pulls in PyTorch).

**✅ Recommended: Google Colab (free GPU, zero setup).** Open this notebook in Colab, then **Runtime → Change runtime type → T4 GPU**. Colab ships a supported Python (3.11 / 3.12) with PyTorch + CUDA already installed, so `ultralytics` installs in seconds and training/inference runs on the GPU.

**💻 Running locally (VS Code / terminal)?** Use **Python 3.11 or 3.12** — Ultralytics + PyTorch do **not** support Python 3.13+/3.14 yet, so `pip install ultralytics` can't find a matching `torch` wheel and fails. Don't change your system Python; make an isolated environment instead:

```bash
# Option A — uv (fast):
uv venv --python 3.12 && source .venv/bin/activate && uv pip install ultralytics

# Option B — conda:
conda create -n yolo python=3.12 -y && conda activate yolo && pip install ultralytics

# verify the install:
python -c "import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())"
```

Training is far faster on a GPU — if you don't have an NVIDIA GPU locally, use Colab.

## 🎯 Learning ObjectivesBy completing this activity, you will be able to:- **Load and configure** pre-trained YOLOv8 models for object detection- **Detect objects** in static images with confidence scores and bounding boxes- **Process video streams** frame-by-frame with real-time object detection- **Analyze detection results** to count specific object classes and filter by confidence- **Understand YOLO architecture** including nano model variants and performance trade-offs

## 📍 Where to run what

This .ipynb is the **canonical edit surface** — work the TODOs in the code cells below; the lesson cells (text + working examples) are read-only context.

YOLOv8 has **three I/O modes** with different runtime requirements:

| Section | Cells | Runtime | Why |
|---|---|---|---|
| Setup + Working Examples 1–2 (install / load model) | 4, 6, 8, 11 | **Colab OR local Python** | `pip install ultralytics`, `YOLO('yolov8n.pt')` work anywhere. |
| Working Example 3 (image detection) | 14 | **Colab-preferred** | Cell ends with `from google.colab.patches import cv2_imshow` for display. Locally, swap that line for `cv.imshow('result', img); cv.waitKey(0)` or use matplotlib. |
| Working Example 4 (video processing) | 17 | **Colab OR local Python** | Just processes frames — doesn't display them. |
| TODO 1 (real-time webcam) | 20 | **Local Python ONLY** | Uses `cv.VideoCapture(0)` + `cv.imshow()` window — Colab has no camera + no `cv.imshow` support. |
| TODO 2 (count by class) | 23 | **Colab OR local Python** | Standard image inference. |
| TODO 3 (confidence filter) | 26 | **Colab OR local Python** | Standard image inference. |

**TODOs you'll edit:** Cells 20, 23, and 26 in this `.ipynb`. Look at the `# Your code here:` outline inside each cell.

**About the `.py` files in this folder** (`1-Setup.py`, `2-Model.py`, `3-Image.py`, `4-Video.py`, `5-Webcam.py`, `Advanced-1.py`):
- `1-Setup.py` is a 7-line runnable setup script (just imports + version checks).
- `2-Model.py` through `5-Webcam.py` and `Advanced-1.py` are **per-stage scaffolds** with TODO comments and empty `# Your code here:` blocks — they're not reference solutions. Completed reference solutions live in the sibling `Template Solution/activity-17-yolo/` folder (instructor-only). The canonical edit target is this `.ipynb`. The `.py` files exist so you can run individual stages from the terminal locally (especially `5-Webcam.py`, which is the standalone version of TODO 1).

**Pre-trained assets included in this template folder:** `yolov8n.pt` (model weights), `outing.jpg` / `advanced1.jpg` / `advanced2.jpg` (sample images), `video.mp4` (sample video). Cell 6's "upload via sidebar" instructions are obsolete — files are already next to the notebook.

## 📦 Setup & InstallationRun this cell to install required packages (Google Colab).

In [ ]:
# Install ultralytics for YOLOv8!pip install -q ultralytics opencv-pythonprint("✅ Packages installed successfully!")

## 📥 Download Required FilesDownload the model, images, and video files needed for this activity.

In [ ]:
# Download YOLOv8 model and sample filesimport urllib.requestimport osfiles = {    'yolov8n.pt': 'https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n.pt',    'outing.jpg': 'sample_image_url',  # Replace with actual URLs    'video.mp4': 'sample_video_url'    # Replace with actual URLs}print("Downloading files...")# Note: In practice, files should be uploaded to Colab or provided via Google Driveprint("⚠️ Please upload: yolov8n.pt, outing.jpg, advanced1.jpg, advanced2.jpg, video.mp4")print("✅ Use the file upload button on the left sidebar")

---## WORKING EXAMPLE 1: Environment Setup & Version CheckThis example verifies that Ultralytics YOLOv8 and OpenCV are correctly installed.

In [ ]:
# WORKING EXAMPLE 1: Verify environment setupimport ultralyticsimport cv2 as cvprint("✅ Environment Setup Complete!")print(f"Ultralytics version: {ultralytics.__version__}")print(f"OpenCV version: {cv.__version__}")

**What this does:**- Imports the Ultralytics library (provides YOLOv8)- Imports OpenCV (cv2) for image/video processing- Prints version numbers to confirm installation**Expected output:**```✅ Environment Setup Complete!Ultralytics version: 8.0.58OpenCV version: 4.7.0.72```

---## WORKING EXAMPLE 2: Load YOLO Model & Extract Class NamesThis example loads the pre-trained YOLOv8 nano model and saves the 80 COCO class names.

In [ ]:
# WORKING EXAMPLE 2: Load YOLOv8 model and extract class namesfrom ultralytics import YOLO# Load the pre-trained modelprint("Loading yolov8n.pt...")model = YOLO('yolov8n.pt')print("✅ Model loaded successfully!")# Extract class names (80 COCO classes)class_names = list(model.names.values())# Display all classesprint(f"\n✅ YOLO can detect {len(class_names)} object classes:")print(class_names)# Save to file for referencewith open('class_names.txt', 'w') as f:    for name in class_names:        f.write(name + '\n')print("\n✅ Class names saved to class_names.txt")

**What this does:**- Loads YOLOv8n model (6.5MB, optimized for speed)- Extracts 80 COCO class names (person, bicycle, car, cat, etc.)- Saves class list to text file for reference**Key concepts:**- **YOLOv8n**: Nano variant (smallest, fastest)- **COCO dataset**: 80 common object classes- **model.names**: Dictionary mapping class IDs (0-79) to names

---## WORKING EXAMPLE 3: Detect Objects in ImageThis example loads an image, runs YOLO detection, and draws bounding boxes with labels.

In [ ]:
# ⚠️ Colab-only cell — uses a JavaScript browser shim / Colab-patches helper.
# In VS Code / local Jupyter / any non-Colab environment, this cell will hang
# forever (you'll see "running 28.0s" with no output). If that happens,
# you're not in Colab — skip this cell or open the notebook in Colab.
# ⚠️ Display path is Colab-specific: this cell ends with `cv2_imshow` from
# `google.colab.patches`. In VS Code / local Jupyter you'll get
# `ModuleNotFoundError: No module named 'google'` on the last 2 lines.
# Local swap: replace the `from google.colab.patches...` + `cv2_imshow(img)` lines
# with `cv.imshow('result', img); cv.waitKey(0); cv.destroyAllWindows()`.
# The detection itself (model inference, box drawing, file save) works everywhere.

# WORKING EXAMPLE 3: Object detection on static image
from ultralytics import YOLO
import cv2 as cv

# Load model and class names
model = YOLO('yolov8n.pt')
class_names = list(model.names.values())

# Load test image
img = cv.imread('outing.jpg')

if img is None:
    print("❌ Error: Could not load image. Please upload outing.jpg")
else:
    print("✅ Image loaded successfully!")

    # Run YOLO detection
    results = model(img)
    print(f"✅ Detected {len(results[0].boxes)} objects")

    # Draw bounding boxes and labels
    for box in results[0].boxes:
        # Extract bounding box coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        # Get class ID and confidence
        class_id = int(box.cls[0])
        confidence = box.conf[0]

        # Create label text
        label = f'{class_names[class_id]} {confidence:.2f}'

        # Draw rectangle (green)
        cv.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Draw label text
        cv.putText(img, label, (x1, y1-10), cv.FONT_HERSHEY_SIMPLEX,
                   0.5, (0, 255, 0), 2)

    # Display result (use imshow in local Python, or save for Colab)
    cv.imwrite('detection_result.jpg', img)
    print("✅ Detection complete! Saved to detection_result.jpg")

    # For Colab, display the image
    from google.colab.patches import cv2_imshow
    cv2_imshow(img)

**What this does:**- Loads image from file- Runs YOLO inference to detect objects- Draws green bounding boxes around each detection- Adds labels with class name + confidence score- Saves and displays the annotated image**Key concepts:**- **box.xyxy**: Bounding box coordinates [x1, y1, x2, y2]- **box.cls**: Class ID (0-79)- **box.conf**: Confidence score (0.0-1.0)- **cv.rectangle()**: Draws bounding box- **cv.putText()**: Adds text label

---## WORKING EXAMPLE 4: Process Video Frame-by-FrameThis example processes a video file, applying YOLO detection to each frame.

In [ ]:
# WORKING EXAMPLE 4: Video processing with frame-by-frame detectionfrom ultralytics import YOLOimport cv2 as cv# Load model and class namesmodel = YOLO('yolov8n.pt')class_names = list(model.names.values())# Load video filevideo = cv.VideoCapture('video.mp4')if not video.isOpened():    print("❌ Error: Could not load video. Please upload video.mp4")else:    print("✅ Video loaded successfully!")    frame_count = 0    # Process video frame by frame    while True:        # Read next frame        ret, frame = video.read()        # Break if no more frames        if not ret:            print(f"\n✅ Processed {frame_count} frames")            break        frame_count += 1        # Run YOLO detection on current frame        results = model(frame)        # Draw bounding boxes on frame        for box in results[0].boxes:            x1, y1, x2, y2 = map(int, box.xyxy[0])            class_id = int(box.cls[0])            confidence = box.conf[0]            label = f'{class_names[class_id]} {confidence:.2f}'            cv.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)            cv.putText(frame, label, (x1, y1-10), cv.FONT_HERSHEY_SIMPLEX,                       0.7, (0, 255, 0), 2)        # Display progress every 30 frames        if frame_count % 30 == 0:            print(f"Processed {frame_count} frames...", end='\r')        # Note: In local Python, use cv.imshow() and waitKey(1) for playback        # In Colab, we just process frames (display not supported)    # Release video capture    video.release()    print("✅ Video processing complete!")

**What this does:**- Opens video file using VideoCapture- Loops through each frame sequentially- Applies YOLO detection to every frame- Draws bounding boxes and labels- Tracks processing progress**Key concepts:**- **video.read()**: Returns (success, frame)- **Frame-by-frame processing**: Video = sequence of images- **Real-time inference**: YOLO fast enough for video (5-30 FPS)- **Resource cleanup**: Always call `video.release()`**Performance:**- CPU: ~100-200ms per frame (5-10 FPS)- GPU: ~30-50ms per frame (30+ FPS)

---## TODO 1: Real-Time Webcam Detection (Medium Difficulty)**Estimated Time**: 15 minutes**What You'll Build:**Use your computer's webcam to perform live object detection, showing real-time bounding boxes and labels as you move objects in front of the camera.**Success Criteria:**- ✅ Webcam stream initiated successfully- ✅ Live video feed displayed on screen- ✅ Objects detected in real-time as they enter frame- ✅ Bounding boxes track moving objects- ✅ Pressing 'Q' stops webcam and closes window**Hints:**- 💡 Access webcam: `cv.VideoCapture(0)` (0 = default camera)- 💡 Check camera opened: `if not video.isOpened(): print("Error")`- 💡 Read frames: Same as video file (`ret, frame = video.read()`)- 💡 Detect objects: `results = model(frame)`- 💡 Draw and display: Same pattern as WORKING EXAMPLE 4- 💡 Font size: Use `0.7` for webcam (matches video)- 💡 Exit condition: `if cv.waitKey(1) & 0xFF == ord('q'): break`- 💡 Cleanup: `video.release()` + `cv.destroyAllWindows()`- 💡 **Important**: Webcam doesn't work in Colab - run locally!**Expected Output:**Live webcam window showing real-time detections with bounding boxes following moving objects.

In [ ]:
# ⚠️ Local Python ONLY — webcam access does NOT work in Google Colab.
# In Colab, `cv.VideoCapture(0)` returns a non-opened object and `cv.imshow()`
# raises a DisabledFunctionError. Run this notebook locally (VS Code Jupyter,
# JupyterLab, or `python 5-Webcam.py` from terminal) for this TODO.

# TODO 1: Real-time webcam detection
# Your code here:
# 1. Import YOLO and OpenCV
# 2. Load model and class names
# 3. Open webcam (VideoCapture with index 0)
# 4. Check if webcam opened successfully
# 5. Loop to read frames from webcam
    # 6. Read frame from webcam
    # 7. Break if frame read failed
    # 8. Run YOLO detection on frame
    # 9. Draw bounding boxes and labels (loop through results[0].boxes)
    # 10. Display frame with detections
    # 11. Check if 'q' key pressed to exit
# 12. Release webcam and close windows

**Note**: Webcam access requires local Python execution. This won't work in Google Colab (no camera access).

---## TODO 2: Count Objects by Class (Hard Difficulty)**Estimated Time**: 20 minutes**What You'll Build:**Load an image, detect all objects, and count how many instances of specific classes (e.g., cats, people) appear in the image.**Success Criteria:**- ✅ Image loaded from `advanced1.jpg`- ✅ All objects detected with YOLO- ✅ Count of "person" instances printed- ✅ Count of "cat" instances printed- ✅ Bounding boxes drawn with counts displayed- ✅ Code works for `advanced2.jpg` when path changed**Hints:**- 💡 Initialize counters: `cat_count = 0`, `person_count = 0`- 💡 Loop through detections: `for box in results[0].boxes:`- 💡 Get class name: `class_name = class_names[int(box.cls[0])]`- 💡 Increment counter: `if class_name == 'cat': cat_count += 1`- 💡 Print totals after loop: `print(f"Cats detected: {cat_count}")`- 💡 Draw boxes same as WORKING EXAMPLE 3- 💡 Test both images: Change path to `advanced2.jpg` and re-run- 💡 COCO classes: "cat" and "person" are standard classes**Expected Output:**```Cats detected: 3People detected: 2```Image displayed with bounding boxes.

In [ ]:
# TODO 2: Count specific object classes# Your code here:# 1. Import necessary packages# 2. Load model and class names# 3. Load image from 'advanced1.jpg'# 4. Run YOLO detection# 5. Initialize counters for cats and people# 6. Loop through detections    # 7. Get class name for this detection    # 8. Increment counters if class matches    # 9. Draw bounding box and label# 10. Print final counts# 11. Display or save the annotated image# 12. (Extension) Change image to 'advanced2.jpg' and re-run

**Extension Ideas:**- Count all 80 classes: Create dictionary of class counts- Filter by confidence: Only count detections above 0.5 confidence- Create report: Save counts to CSV file- Visualize: Create bar chart of object counts

---## TODO 3: Filter Detections by Confidence Threshold (Medium Difficulty)**Estimated Time**: 15 minutes**What You'll Build:**Load an image and only display detections that exceed a specified confidence threshold (e.g., 0.7 = 70% confident).**Success Criteria:**- ✅ Image loaded successfully- ✅ YOLO detection runs with confidence filtering- ✅ Only high-confidence detections displayed (e.g., >0.7)- ✅ Lower confidence threshold shows more objects- ✅ Higher confidence threshold shows fewer, but more accurate detections**Hints:**- 💡 Load image and run detection as usual- 💡 Set threshold: `confidence_threshold = 0.7`- 💡 Filter in loop: `if box.conf[0] < confidence_threshold: continue`- 💡 Only draw boxes that pass threshold- 💡 Try different thresholds: 0.3, 0.5, 0.7, 0.9- 💡 YOLO default threshold is 0.25- 💡 Higher threshold = fewer false positives, may miss some objects- 💡 Lower threshold = more detections, may include false positives**Expected Output:**Image with only high-confidence detections (fewer bounding boxes than default).

In [ ]:
# TODO 3: Filter detections by confidence threshold# Your code here:# 1. Import packages and load model# 2. Load test image# 3. Run YOLO detection# 4. Set confidence threshold (try 0.7 first)# 5. Loop through detections    # 6. Get confidence score for this detection    # 7. Skip if below threshold    # 8. Draw bounding box and label for high-confidence detections# 9. Display or save filtered results# 10. (Extension) Try different thresholds and compare results

**Extension Challenge:**- Create a function that takes an image and threshold as input- Process same image with thresholds: 0.3, 0.5, 0.7, 0.9- Save 4 output images showing how threshold affects results- Count number of detections at each threshold level

---## 🎉 Congratulations!Once you complete this activity, you'll have:- ✅ **Mastered YOLOv8** for real-time object detection- ✅ **Processed images and videos** with computer vision AI- ✅ **Built practical skills** for surveillance, counting, tracking applications- ✅ **Portfolio project** showcasing state-of-the-art object detection**Real-world applications:**- 🚗 Autonomous vehicles (detecting pedestrians, cars, traffic signs)- 📹 Security systems (perimeter monitoring, crowd counting)- 🏭 Manufacturing (quality control, defect detection)- 🏥 Healthcare (medical image analysis, patient monitoring)- 🛒 Retail (inventory tracking, customer analytics)

## 📚 Additional Resources**Documentation:**- [Ultralytics YOLOv8 Docs](https://docs.ultralytics.com/)- [OpenCV Python Tutorials](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)- [COCO Dataset](https://cocodataset.org/)**Related Activities:**- Activity 16: Image Classification with CNNs (prerequisite)- Activity 18: Semantic Segmentation (next topic)- Activities 13-15: OpenCV Fundamentals